# ASL letter recognition: full frame versus automatic hand crop

This notebook runs two real evaluations using the same pretrained 26-label EfficientNet-B0 classifier: full original image (baseline) and MediaPipe automatic hand crop.

Only 12 static letters are evaluated, but predictions remain over all 26 model labels. A crop detection failure counts as incorrect. The public image split does not establish signer independence.

### 1. Load the project from the GitHub fork

The first code cell clones the public fork if needed, then enters the ASL experiment subdirectory. If you already cloned the fork before the experiment was added, first run `cd /content/cs117-asl-ablation && git pull --ff-only` in the terminal.

In [ ]:
from pathlib import Path
import os
import subprocess

checkout = Path("/content/cs117-asl-ablation")
project = checkout / "examples/asl_letter_ablation"
if not (project / "scripts/evaluate.py").is_file():
    if checkout.exists():
        raise RuntimeError("The fork checkout exists but lacks the experiment. In the terminal run: cd /content/cs117-asl-ablation && git pull --ff-only")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/haddadmf/cs117-asl-ablation.git", str(checkout)], check=True)
if not (project / "scripts/evaluate.py").is_file():
    raise FileNotFoundError("ASL experiment not found inside fork.")
os.chdir(project)
print("Experiment working directory:", Path.cwd())

### 2. Install dependencies and run sanity tests (not a baseline measurement)

In [ ]:
%pip -q install -r requirements.txt
!python -m unittest discover -s tests -v

### 3. Download original photos and prepare the split manifest

This step downloads the public ASL Letters dataset in Pascal VOC format. If its URL changes, manually obtain the VOC ZIP from https://datasets.cms.waikato.ac.nz/ufdl/american-sign-language-letters/ and place it in `data/raw/asl_letters_voc.zip` before preparing the manifest.

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_dataset.py
from pathlib import Path
import pandas as pd
manifest = pd.read_csv('data/processed/manifest.csv')
display(manifest.groupby(['split','label']).size().unstack(fill_value=0))
print('Important: This dataset does NOT provide a verified signer-disjoint split.')

### 4. Measure full-frame top-1 accuracy on validation photos

The fixed pretrained checkpoint downloads on first use. This cell writes an actual measured result and per-image predictions, unlike the sanity tests.

In [ ]:
!python scripts/evaluate.py --split val --method full --output results/baseline_val.json
import json
from pathlib import Path
b = json.loads(Path("results/baseline_val.json").read_text())
print("Baseline top-1 accuracy:", f"{b['top1_accuracy_pct']}%")
print("Correct / evaluated:", f"{b['correct']}/{b['images_evaluated']}")
print("Split:", b["split"])
print("Proposal estimate was 65%; measured result is", "above" if b["top1_accuracy_pct"] > 65 else ("below" if b["top1_accuracy_pct"] < 65 else "equal to"), "that estimate.")

### 5. Real-image classifier and MediaPipe smoke test

In [ ]:
!python scripts/smoke_test.py --check-crops --output results/smoke_test.txt
print(Path('results/smoke_test.txt').read_text())

### 6. Automatic hand-crop evaluation on the same validation images

Hand-detection failures count as incorrect; prediction labels remain the same 26 classes.

In [ ]:
!python scripts/evaluate.py --split val --method crop --output results/crop_val.json
!python scripts/report_results.py
import pandas as pd
display(pd.read_csv('results/main_table.csv'))

### 7. Optional background annotation

Generate `docs/background_audit.csv`, then inspect the actual source photos and fill plain/cluttered labels by hand. Run `python scripts/apply_background_sheet.py` to apply manually verified labels and rerun both evaluations. Otherwise do not claim background-specific accuracy.

In [ ]:
!python scripts/make_background_sheet.py --split val
print('Generated docs/background_audit.csv; open it and inspect source images. Do not auto-guess labels.')

### 8. Export measured evidence

This exports only real output files produced by the evaluation. Commit the result JSON and prediction CSV files separately so the experiment is reproducible.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
names = [
    "results/baseline_val.json",
    "results/baseline_val_predictions.csv",
    "results/smoke_test.txt",
    "results/crop_val.json",
    "results/crop_val_predictions.csv",
    "results/main_table.csv",
]
with zipfile.ZipFile("/content/ASL_measured_results.zip", "w") as z:
    for name in names:
        if Path(name).is_file():
            z.write(name)
print("Exported measured results to /content/ASL_measured_results.zip")
files.download("/content/ASL_measured_results.zip")